In [1]:
TICKERS = [
    "AUR",
    "TSLA",
    "MBLY",
    "GOOGL",
    "GM",
    "F",
    "NVDA",
    "QCOM",
    "APTV",
    "OUST",
    "RIVN",
]

In [5]:
from pathlib import Path
import json
import numpy as np

EMBEDDINGS_DIR = Path("../data/embeddings")
CHUNKS_DIR = Path("../data/chunks")

FILINGS = {
    "AUR": "2025-10-K",
    "TSLA": "2025-10-K",
    "MBLY": "2025-10-K",
    "GOOGL": "2025-10-K",
    "GM": "2025-10-K",
    "F": "2025-10-K",
    "NVDA": "2026-10-K",
    "QCOM": "2025-10-K",
    "APTV": "2025-10-K",
    "OUST": "2025-10-K",
    "RIVN": "2025-10-K",
}

company_embeddings = []
all_chunks = []

for ticker, filing_name in FILINGS.items():
    embedding_paths = list(
        (EMBEDDINGS_DIR / ticker).glob(f"{filing_name}.bgebase*.npz")
    )
    if len(embedding_paths) != 1:
        raise ValueError(
            f"Expected one BGE-base artifact for {ticker}; found {embedding_paths}"
        )
    embedding_path = embedding_paths[0]

    data = np.load(embedding_path)
    embeddings = data["embeddings"]

    chunk_path = (
        CHUNKS_DIR / ticker / f"{filing_name}.chunks.jsonl"
    )

    with open(chunk_path, "r", encoding="utf-8") as f:
        chunks = [json.loads(line) for line in f if line.strip()]

    assert len(embeddings) == len(chunks), (
        f"{ticker}: {len(embeddings)} embeddings != {len(chunks)} chunks"
    )

    company_embeddings.append(embeddings)
    all_chunks.extend(chunks)

all_embeddings = np.vstack(company_embeddings)

assert len(all_embeddings) == len(all_chunks)

print("Total chunks:", len(all_chunks))
print("Embedding matrix:", all_embeddings.shape)

Total chunks: 4115
Embedding matrix: (4115, 768)


In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-base-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7310.67it/s]


In [7]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-base")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7571.83it/s]


In [9]:
test_queries = {
    "Aurora Innovation": [
        {
            "ticker": "AUR",
            "expected_type": "table",
            "query": "What were Aurora's research and development expenses in 2025?",
        },
        {
            "ticker": "AUR",
            "expected_type": "narrative",
            "query": "How does Aurora describe its commercial driverless trucking operations?",
        },
    ],

    "Tesla": [
        {
            "ticker": "TSLA",
            "expected_type": "table",
            "query": "What were Tesla's automotive revenues in 2025?",
        },
        {
            "ticker": "TSLA",
            "expected_type": "narrative",
            "query": "Which consumer vehicle models does Tesla currently manufacture?",
        },
    ],

    "Mobileye": [
        {
            "ticker": "MBLY",
            "expected_type": "table",
            "query": "What were Mobileye's total revenues in 2024?",
        },
        {
            "ticker": "MBLY",
            "expected_type": "narrative",
            "query": "How does Mobileye describe its EyeQ SoC business and the role it plays in company revenue?",
        },
    ],

    "Alphabet": [
        {
            "ticker": "GOOGL",
            "expected_type": "table",
            "query": "What were Google Cloud revenues in 2025?",
        },
        {
            "ticker": "GOOGL",
            "expected_type": "narrative",
            "query": "How does Alphabet describe Waymo and its autonomous driving business?",
        },
    ],

    "General Motors": [
        {
            "ticker": "GM",
            "expected_type": "table",
            "query": "What were General Motors' automotive net sales and revenue in 2025?",
        },
        {
            "ticker": "GM",
            "expected_type": "narrative",
            "query": "How does General Motors describe its autonomous vehicle technology strategy?",
        },
    ],

    "Ford": [
        {
            "ticker": "F",
            "expected_type": "table",
            "query": "What were Ford Model e revenues in 2025?",
        },
        {
            "ticker": "F",
            "expected_type": "narrative",
            "query": "How does Ford describe the roles of Ford Blue, Ford Model e, and Ford Pro?",
        },
    ],

    "NVIDIA": [
        {
            "ticker": "NVDA",
            "expected_type": "table",
            "query": "What was NVIDIA Data Center revenue in fiscal 2026?",
        },
        {
            "ticker": "NVDA",
            "expected_type": "narrative",
            "query": "How does NVIDIA describe its automotive and autonomous-driving computing platform?",
        },
    ],

    "Qualcomm": [
        {
            "ticker": "QCOM",
            "expected_type": "table",
            "query": "What were Qualcomm automotive revenues in 2025?",
        },
        {
            "ticker": "QCOM",
            "expected_type": "narrative",
            "query": "How does Qualcomm describe its automotive business and Snapdragon Digital Chassis?",
        },
    ],

    "Aptiv": [
        {
            "ticker": "APTV",
            "expected_type": "table",
            "query": "What were Aptiv's net sales by business segment in 2025?",
        },
        {
            "ticker": "APTV",
            "expected_type": "narrative",
            "query": "What products and technologies does Aptiv provide for advanced safety and vehicle user experience?",
        },
    ],

    "Ouster": [
        {
            "ticker": "OUST",
            "expected_type": "table",
            "query": "What was Ouster's revenue in 2025?",
        },
        {
            "ticker": "OUST",
            "expected_type": "narrative",
            "query": "How does Ouster describe its lidar products and the markets they serve?",
        },
    ],

    "Rivian": [
        {
            "ticker": "RIVN",
            "expected_type": "table",
            "query": "How much total revenue did Rivian report in 2025, and how was it split by major source?",
        },
        {
            "ticker": "RIVN",
            "expected_type": "narrative",
            "query": "How does Rivian describe its R1, R2, and R3 vehicle platforms?",
        },
    ],
}

In [10]:
import numpy as np


def retrieve_candidates(
    query,
    model,
    all_embeddings,
    all_chunks,
    top_k=30,
    table_top_k=10,
):
    query_text = f"Represent this sentence for searching relevant passages: {query}"

    query_embedding = model.encode(
        query_text,
        normalize_embeddings=True,
    )

    # Normalize stored embeddings
    norms = np.linalg.norm(all_embeddings, axis=1, keepdims=True)
    normalized_embeddings = all_embeddings / np.clip(norms, 1e-12, None)

    # -------------------------------------------------
    # 1. Global retrieval
    # -------------------------------------------------
    scores = normalized_embeddings @ query_embedding

    global_indices = np.argsort(scores)[-top_k:][::-1]

    # -------------------------------------------------
    # 2. Table-only retrieval
    # -------------------------------------------------
    table_indices = [
        i
        for i, chunk in enumerate(all_chunks)
        if chunk.get("content_type") == "table"
    ]

    table_top_indices = []

    if table_indices:
        table_embeddings = normalized_embeddings[table_indices]
        table_scores = table_embeddings @ query_embedding

        local_indices = np.argsort(table_scores)[-table_top_k:][::-1]

        table_top_indices = [
            table_indices[i]
            for i in local_indices
        ]

    # -------------------------------------------------
    # 3. Merge + deduplicate
    # -------------------------------------------------
    candidate_indices = []

    seen = set()

    for idx in list(global_indices) + table_top_indices:
        if idx not in seen:
            candidate_indices.append(idx)
            seen.add(idx)

    # -------------------------------------------------
    # 4. Build candidate objects
    # -------------------------------------------------
    results = []

    for idx in candidate_indices:
        results.append({
            "index": idx,
            "retrieval_score": float(scores[idx]),
            "chunk": all_chunks[idx],
        })

    return results

In [11]:
def print_table(chunk):
    title = chunk.get("title")
    headers = chunk.get("table_headers") or chunk.get("logical_header_rows")
    rows = chunk.get("table_rows") or chunk.get("logical_rows")

    if title:
        print(f"TABLE: {title}")

    if chunk.get("units"):
        print(f"Units: {chunk['units']}")

    print()

    if headers and rows:
        # table_headers is normally list[list]
        header = headers[-1] if isinstance(headers[0], list) else headers

        table = [header] + rows

        table = [
            ["" if cell is None else str(cell) for cell in row]
            for row in table
        ]

        col_count = max(len(row) for row in table)

        widths = [
            max(
                len(row[col]) if col < len(row) else 0
                for row in table
            )
            for col in range(col_count)
        ]

        def format_row(row):
            return " | ".join(
                (
                    row[i] if i < len(row) else ""
                ).ljust(widths[i])
                for i in range(col_count)
            )

        print(format_row(table[0]))
        print("-+-".join("-" * width for width in widths))

        for row in table[1:]:
            print(format_row(row))

    else:
        # Your chunk schema already has the full Markdown table here
        print(chunk.get("text", ""))

In [12]:
def print_results(results, narrative_chars=500):
    for rank, result in enumerate(results, start=1):
        chunk = result["chunk"]

        print("=" * 120)

        print(
            f"{rank:2}. "
            f"{chunk.get('ticker')} | "
            f"{result['retrieval_score']:.4f} | "
            f"type={chunk.get('content_type')} | "
            f"{chunk.get('section')}"
        )

        print("=" * 120)

        if chunk.get("content_type") == "table":
            print_table(chunk)

        else:
            print(chunk.get("text", "")[:narrative_chars])

        print()

In [13]:
query = "Mobileye revenue 2024"

results = retrieve_candidates(
    query=query,
    model=model,
    all_embeddings=all_embeddings,
    all_chunks=all_chunks,
    top_k=30,
    table_top_k=10,
)

print_results(results)

 1. MBLY | 0.6844 | type=table | Item 8 — Financial Statements
Units: usd_millions

Item 8 — Financial Statements
NOTES TO CONSOLIDATED FINANCIAL STATEMENTS
Financial statement notes

Units: usd_millions

| Line item | Year ended December 28, 2024 — Mobileye | Year ended December 28, 2024 — Other | Year ended December 28, 2024 — Total | Year ended December 30, 2023 — Mobileye | Year ended December 30, 2023 — Other | Year ended December 30, 2023 — Total |
| :--- | ---: | ---: | ---: | ---: | ---: | ---: |
| Revenues | $1,613 | $41 | $1,654 | $2,045 | $34 | $2,079 |
| Cost of revenues | 529 | 6 | — | 619 | 5 | — |
| Research and development, net | 810 | 29 | — | 645 | 32 | — |
| Sales and marketing | 31 | 13 | — | 33 | 10 | — |
| General and administrative | 40 | 3 | — | 38 | 4 | — |
| Segment performance | $203 | $(10) | $193 | $710 | $(17) | $693 |
| Amortization of intangible assets |  |  | (444) |  |  | (474) |
| Share-based compensation |  |  | (279) |  |  | (252) |
| Goodwill impai

In [14]:
import numpy as np

from src.embeddings.embed_chunks import table_embedding_text


# ============================================================
# PRECOMPUTE NORMALIZED CORPUS EMBEDDINGS ONCE
# ============================================================

embedding_norms = np.linalg.norm(
    all_embeddings,
    axis=1,
    keepdims=True,
)

normalized_embeddings = (
    all_embeddings / np.clip(embedding_norms, 1e-12, None)
)


# ============================================================
# RERANKER INPUT
# ============================================================

def get_rerank_text(chunk):
    """
    Narrative:
        explicit company metadata + original chunk text

    Table:
        explicit company metadata
        + compact semantic table representation
        + full original Markdown table

    The compact representation comes first so the important
    semantic information is retained even if the cross-encoder
    truncates a very large table.
    """

    prefix = (
        f"Company: {chunk.get('company', '')}\n"
        f"Ticker: {chunk.get('ticker', '')}\n"
        f"Section: {chunk.get('section', '')}\n"
        f"Content type: {chunk.get('content_type', '')}\n"
    )

    if chunk.get("content_type") == "table":
        compact_table = table_embedding_text(chunk)

        return (
            prefix
            + "\n"
            + compact_table
            + "\n\nFull table:\n"
            + chunk.get("text", "")
        )

    return prefix + "\n" + chunk.get("text", "")


# ============================================================
# RETRIEVAL + RERANKING
# ============================================================

def retrieve_candidates(
    query,
    model,
    reranker,
    normalized_embeddings,
    all_chunks,
    top_k=30,
    table_top_k=10,
    final_top_k=10,
    reranker_batch_size=32,
):
    # --------------------------------------------------------
    # 1. Embed query
    # --------------------------------------------------------

    query_text = (
        "Represent this sentence for searching relevant passages: "
        + query
    )

    query_embedding = model.encode(
        query_text,
        normalize_embeddings=True,
    )

    # --------------------------------------------------------
    # 2. Global dense retrieval
    # --------------------------------------------------------

    dense_scores = normalized_embeddings @ query_embedding

    global_indices = np.argsort(
        dense_scores
    )[-top_k:][::-1]

    global_rank = {
        int(idx): rank
        for rank, idx in enumerate(global_indices, start=1)
    }

    # --------------------------------------------------------
    # 3. Table-only dense retrieval
    # --------------------------------------------------------

    table_indices = np.array([
        i
        for i, chunk in enumerate(all_chunks)
        if chunk.get("content_type") == "table"
    ])

    table_top_indices = []
    table_rank = {}

    if len(table_indices) > 0:
        table_embeddings = normalized_embeddings[table_indices]

        table_scores = table_embeddings @ query_embedding

        local_indices = np.argsort(
            table_scores
        )[-table_top_k:][::-1]

        table_top_indices = [
            int(table_indices[i])
            for i in local_indices
        ]

        table_rank = {
            idx: rank
            for rank, idx in enumerate(
                table_top_indices,
                start=1,
            )
        }

    # --------------------------------------------------------
    # 4. Merge global + table candidates and deduplicate
    # --------------------------------------------------------

    candidate_indices = []
    seen = set()

    for idx in list(global_indices) + table_top_indices:
        idx = int(idx)

        if idx not in seen:
            candidate_indices.append(idx)
            seen.add(idx)

    # --------------------------------------------------------
    # 5. Build candidate objects
    # --------------------------------------------------------

    results = []

    for idx in candidate_indices:
        chunk = all_chunks[idx]

        results.append({
            "index": idx,
            "chunk": chunk,
            "dense_score": float(dense_scores[idx]),
            "global_dense_rank": global_rank.get(idx),
            "table_dense_rank": table_rank.get(idx),
            "from_global": idx in global_rank,
            "from_table_search": idx in table_rank,
        })

    # --------------------------------------------------------
    # 6. Cross-encoder reranking
    # --------------------------------------------------------

    pairs = [
        (
            query,
            get_rerank_text(result["chunk"]),
        )
        for result in results
    ]

    reranker_scores = reranker.predict(
        pairs,
        batch_size=reranker_batch_size,
        show_progress_bar=False,
    )

    for result, score in zip(results, reranker_scores):
        result["reranker_score"] = float(score)

    # --------------------------------------------------------
    # 7. Final reranked ordering
    # --------------------------------------------------------

    results.sort(
        key=lambda x: x["reranker_score"],
        reverse=True,
    )

    for rank, result in enumerate(results, start=1):
        result["rerank"] = rank

    return results[:final_top_k]


# ============================================================
# PRINT RESULTS
# ============================================================

def print_results(results):
    for result in results:
        chunk = result["chunk"]

        print("=" * 120)

        print(
            f"{result['rerank']:2}. "
            f"{chunk.get('ticker')} | "
            f"type={chunk.get('content_type')} | "
            f"dense={result['dense_score']:.4f} | "
            f"rerank={result['reranker_score']:.4f}"
        )

        print(
            f"global_rank={result['global_dense_rank']} | "
            f"table_rank={result['table_dense_rank']} | "
            f"global={result['from_global']} | "
            f"table_search={result['from_table_search']}"
        )

        print(
            f"Section: {chunk.get('section')}"
        )

        print("-" * 120)

        # Print complete tables so values can be inspected.
        if chunk.get("content_type") == "table":
            print(chunk.get("text", ""))

        # Narrative output can stay truncated for readability.
        else:
            print(chunk.get("text", "")[:2000])

        print()

In [16]:
company = "Mobileye"
test_case = test_queries[company][0]  # 0 = table, 1 = narrative

query = test_case["query"]

results = retrieve_candidates(
    query=query,
    model=model,
    reranker=reranker,
    normalized_embeddings=normalized_embeddings,
    all_chunks=all_chunks,
    top_k=15,
    table_top_k=5,
    final_top_k=10,
)

print(f"Company:       {company}")
print(f"Expected type: {test_case['expected_type']}")
print(f"Query:         {query}\n")

print_results(results)

Company:       Mobileye
Expected type: table
Query:         What were Mobileye's total revenues in 2024?

 1. MBLY | type=table | dense=0.6849 | rerank=0.9994
global_rank=1 | table_rank=1 | global=True | table_search=True
Section: Item 8 — Financial Statements
------------------------------------------------------------------------------------------------------------------------
Item 8 — Financial Statements
NOTES TO CONSOLIDATED FINANCIAL STATEMENTS
Financial statement notes

Units: usd_millions

| Line item | Year ended December 28, 2024 — Mobileye | Year ended December 28, 2024 — Other | Year ended December 28, 2024 — Total | Year ended December 30, 2023 — Mobileye | Year ended December 30, 2023 — Other | Year ended December 30, 2023 — Total |
| :--- | ---: | ---: | ---: | ---: | ---: | ---: |
| Revenues | $1,613 | $41 | $1,654 | $2,045 | $34 | $2,079 |
| Cost of revenues | 529 | 6 | — | 619 | 5 | — |
| Research and development, net | 810 | 29 | — | 645 | 32 | — |
| Sales and market

In [136]:
import time
import numpy as np

from src.embeddings.embed_chunks import table_embedding_text


# ============================================================
# NORMALIZE CORPUS EMBEDDINGS ONCE
# ============================================================

norms = np.linalg.norm(all_embeddings, axis=1, keepdims=True)
normalized_embeddings = all_embeddings / np.clip(norms, 1e-12, None)


# ============================================================
# BUILD TEXT FOR CROSS-ENCODER
# ============================================================

def get_rerank_text(chunk):
    prefix = (
        f"Company: {chunk.get('company', '')}\n"
        f"Ticker: {chunk.get('ticker', '')}\n"
        f"Section: {chunk.get('section', '')}\n"
        f"Content type: {chunk.get('content_type', '')}\n"
    )

    if chunk.get("content_type") == "table":
        compact_table = table_embedding_text(chunk)

        return (
            prefix
            + "\n"
            + compact_table
            + "\n\nFull table:\n"
            + chunk.get("text", "")
        )

    return prefix + "\n" + chunk.get("text", "")


# ============================================================
# RETRIEVE + RERANK
# ============================================================

def retrieve_candidates(
    query,
    global_top_k=30,
    table_top_k=10,
    final_top_k=10,
    batch_size=32,
):
    total_start = time.perf_counter()

    # --------------------------------------------------------
    # 1. QUERY EMBEDDING
    # --------------------------------------------------------

    t0 = time.perf_counter()

    retrieval_query = (
        "Represent this sentence for searching relevant passages: "
        + query
    )

    query_embedding = model.encode(
        retrieval_query,
        normalize_embeddings=True,
    )

    query_embedding_time = time.perf_counter() - t0

    # --------------------------------------------------------
    # 2. GLOBAL DENSE RETRIEVAL
    # --------------------------------------------------------

    t0 = time.perf_counter()

    dense_scores = normalized_embeddings @ query_embedding

    global_indices = np.argsort(
        dense_scores
    )[-global_top_k:][::-1]

    global_rank = {
        int(idx): rank
        for rank, idx in enumerate(global_indices, start=1)
    }

    # --------------------------------------------------------
    # 3. TABLE-ONLY DENSE RETRIEVAL
    # --------------------------------------------------------

    table_indices = np.array([
        i
        for i, chunk in enumerate(all_chunks)
        if chunk.get("content_type") == "table"
    ])

    table_selected_indices = []
    table_rank = {}

    if len(table_indices) > 0:
        table_embeddings = normalized_embeddings[table_indices]

        table_scores = table_embeddings @ query_embedding

        local_table_indices = np.argsort(
            table_scores
        )[-table_top_k:][::-1]

        table_selected_indices = [
            int(table_indices[i])
            for i in local_table_indices
        ]

        table_rank = {
            idx: rank
            for rank, idx in enumerate(
                table_selected_indices,
                start=1,
            )
        }

    dense_retrieval_time = time.perf_counter() - t0

    # --------------------------------------------------------
    # 4. MERGE + DEDUPLICATE
    # --------------------------------------------------------

    candidate_indices = []
    seen = set()

    for idx in list(global_indices) + table_selected_indices:
        idx = int(idx)

        if idx not in seen:
            candidate_indices.append(idx)
            seen.add(idx)

    results = []

    for idx in candidate_indices:
        results.append({
            "index": idx,
            "chunk": all_chunks[idx],
            "dense_score": float(dense_scores[idx]),
            "global_dense_rank": global_rank.get(idx),
            "table_dense_rank": table_rank.get(idx),
            "from_global": idx in global_rank,
            "from_table_search": idx in table_rank,
        })

    # --------------------------------------------------------
    # 5. PREPARE RERANKER INPUT
    # --------------------------------------------------------

    pairs = [
        (
            query,
            get_rerank_text(result["chunk"]),
        )
        for result in results
    ]

    # Measure actual token lengths fed to reranker
    encoded = reranker.tokenizer(
        [q for q, _ in pairs],
        [text for _, text in pairs],
        truncation=True,
        max_length=512,
    )

    token_lengths = [
        len(ids)
        for ids in encoded["input_ids"]
    ]

    # --------------------------------------------------------
    # 6. CROSS-ENCODER RERANKING
    # --------------------------------------------------------

    t0 = time.perf_counter()

    reranker_scores = reranker.predict(
        pairs,
        batch_size=batch_size,
        show_progress_bar=False,
    )

    reranker_time = time.perf_counter() - t0

    for result, score in zip(results, reranker_scores):
        result["reranker_score"] = float(score)

    results.sort(
        key=lambda x: x["reranker_score"],
        reverse=True,
    )

    for rank, result in enumerate(results, start=1):
        result["rerank"] = rank

    total_time = time.perf_counter() - total_start

    # --------------------------------------------------------
    # 7. DIAGNOSTICS
    # --------------------------------------------------------

    print("\n--- Retrieval diagnostics ---")
    print(f"Global candidates requested: {global_top_k}")
    print(f"Table candidates requested:  {table_top_k}")
    print(f"Candidates after dedup:      {len(results)}")

    print("\n--- Reranker input ---")
    print(f"Average tokens: {np.mean(token_lengths):.1f}")
    print(f"Max tokens:     {max(token_lengths)}")
    print(f"Total tokens:   {sum(token_lengths)}")

    print("\n--- Timing ---")
    print(f"Query embedding: {query_embedding_time:.3f}s")
    print(f"Dense retrieval: {dense_retrieval_time:.3f}s")
    print(f"Reranking:       {reranker_time:.3f}s")
    print(f"Total:           {total_time:.3f}s")

    return results[:final_top_k]


# ============================================================
# PRINT FINAL RESULTS
# ============================================================

def print_results(results, narrative_chars=2000):
    for result in results:
        chunk = result["chunk"]

        print("=" * 120)

        print(
            f"{result['rerank']:2}. "
            f"{chunk.get('ticker')} | "
            f"type={chunk.get('content_type')} | "
            f"dense={result['dense_score']:.4f} | "
            f"rerank={result['reranker_score']:.4f}"
        )

        print(
            f"global_rank={result['global_dense_rank']} | "
            f"table_rank={result['table_dense_rank']} | "
            f"global={result['from_global']} | "
            f"table_search={result['from_table_search']}"
        )

        print(f"Section: {chunk.get('section')}")
        print("-" * 120)

        if chunk.get("content_type") == "table":
            # Entire table
            print(chunk.get("text", ""))
        else:
            print(chunk.get("text", "")[:narrative_chars])

        print()

In [138]:
company = "NVIDIA"
test_case = test_queries[company][0]  # 0 = table, 1 = narrative

query = test_case["query"]

print(f"Company:       {company}")
print(f"Expected type: {test_case['expected_type']}")
print(f"Query:         {query}")

results = retrieve_candidates(
    query=query,
    global_top_k=20,
    table_top_k=5,
    final_top_k=10,
    batch_size=32,
)

print_results(results)

Company:       NVIDIA
Expected type: table
Query:         What was NVIDIA Data Center revenue in fiscal 2026?

--- Retrieval diagnostics ---
Global candidates requested: 20
Table candidates requested:  5
Candidates after dedup:      25

--- Reranker input ---
Average tokens: 327.0
Max tokens:     512
Total tokens:   8176

--- Timing ---
Query embedding: 0.111s
Dense retrieval: 0.011s
Reranking:       14.785s
Total:           14.924s
 1. NVDA | type=narrative | dense=0.7223 | rerank=0.9999
global_rank=1 | table_rank=None | global=True | table_search=False
Section: Item 7 — Management's Discussion and Analysis of Financial Condition and Results of Operations
------------------------------------------------------------------------------------------------------------------------
Item 7 — Management's Discussion and Analysis of Financial Condition and Results of Operations
Fiscal Year 2026 Summary

Revenue for fiscal year 2026 was $215.9 billion, up 65% from a year ago.

Data Center revenue

In [134]:
import time
import numpy as np


# ============================================================
# PURE RETRIEVAL
# ============================================================

def retrieve_candidates(
    query,
    global_top_k=30,
    table_top_k=10,
):
    start = time.perf_counter()

    retrieval_query = (
        "Represent this sentence for searching relevant passages: "
        + query
    )

    query_embedding = model.encode(
        retrieval_query,
        normalize_embeddings=True,
    )

    dense_scores = normalized_embeddings @ query_embedding

    # Global retrieval
    global_indices = np.argsort(
        dense_scores
    )[-global_top_k:][::-1]

    global_rank = {
        int(idx): rank
        for rank, idx in enumerate(global_indices, start=1)
    }

    # Table-only retrieval
    table_indices = np.array([
        i
        for i, chunk in enumerate(all_chunks)
        if chunk.get("content_type") == "table"
    ])

    table_selected_indices = []
    table_rank = {}

    if len(table_indices) > 0:
        table_scores = (
            normalized_embeddings[table_indices]
            @ query_embedding
        )

        local_indices = np.argsort(
            table_scores
        )[-table_top_k:][::-1]

        table_selected_indices = [
            int(table_indices[i])
            for i in local_indices
        ]

        table_rank = {
            idx: rank
            for rank, idx in enumerate(
                table_selected_indices,
                start=1,
            )
        }

    # Merge + deduplicate
    candidate_indices = []
    seen = set()

    for idx in list(global_indices) + table_selected_indices:
        idx = int(idx)

        if idx not in seen:
            candidate_indices.append(idx)
            seen.add(idx)

    results = []

    for idx in candidate_indices:
        results.append({
            "index": idx,
            "chunk": all_chunks[idx],
            "dense_score": float(dense_scores[idx]),
            "global_dense_rank": global_rank.get(idx),
            "table_dense_rank": table_rank.get(idx),
            "from_global": idx in global_rank,
            "from_table_search": idx in table_rank,
        })

    retrieval_time = time.perf_counter() - start

    return results, retrieval_time


# ============================================================
# RERANKING
# ============================================================

def rerank_candidates(
    query,
    candidates,
    final_top_k=10,
    batch_size=32,
):
    start = time.perf_counter()

    pairs = [
        (
            query,
            get_rerank_text(result["chunk"]),
        )
        for result in candidates
    ]

    # Diagnostics
    encoded = reranker.tokenizer(
        [q for q, _ in pairs],
        [text for _, text in pairs],
        truncation=True,
        max_length=512,
    )

    token_lengths = [
        len(ids)
        for ids in encoded["input_ids"]
    ]

    # Cross-encoder
    reranker_scores = reranker.predict(
        pairs,
        batch_size=batch_size,
        show_progress_bar=False,
    )

    results = []

    for candidate, score in zip(candidates, reranker_scores):
        result = candidate.copy()
        result["reranker_score"] = float(score)
        results.append(result)

    results.sort(
        key=lambda x: x["reranker_score"],
        reverse=True,
    )

    for rank, result in enumerate(results, start=1):
        result["rerank"] = rank

    reranking_time = time.perf_counter() - start

    diagnostics = {
        "candidate_count": len(results),
        "average_tokens": float(np.mean(token_lengths)),
        "max_tokens": max(token_lengths),
        "total_tokens": sum(token_lengths),
        "reranking_time": reranking_time,
    }

    return results[:final_top_k], diagnostics

In [135]:
query = "What were Mobileye's total revenues in 2024?"

candidates, retrieval_time = retrieve_candidates(
    query=query,
    global_top_k=30,
    table_top_k=10,
)
print_results(candidates)

KeyError: 'rerank'

In [ ]:
def print_pure_retrieval_results(candidates, top_k=10, narrative_chars=2000):
    results = sorted(
        candidates,
        key=lambda x: x["dense_score"],
        reverse=True,
    )[:top_k]

    for rank, result in enumerate(results, start=1):
        chunk = result["chunk"]

        print("=" * 120)

        print(
            f"{rank:2}. "
            f"{chunk.get('ticker')} | "
            f"type={chunk.get('content_type')} | "
            f"dense={result['dense_score']:.4f}"
        )

        print(
            f"global_rank={result.get('global_dense_rank')} | "
            f"table_rank={result.get('table_dense_rank')} | "
            f"global={result.get('from_global')} | "
            f"table_search={result.get('from_table_search')}"
        )

        print(f"Section: {chunk.get('section')}")
        print("-" * 120)

        if chunk.get("content_type") == "table":
            print(chunk.get("text", ""))
        else:
            print(chunk.get("text", "")[:narrative_chars])

        print()

In [119]:
# ============================================================
# PURE DENSE TOP-K FROM MERGED CANDIDATE POOL
# ============================================================

def get_dense_top_k(candidates, top_k=10):
    results = [
        candidate.copy()
        for candidate in candidates
    ]

    results.sort(
        key=lambda x: x["dense_score"],
        reverse=True,
    )

    for rank, result in enumerate(results, start=1):
        result["dense_final_rank"] = rank

    return results[:top_k]


# ============================================================
# PRINT COMPACT RESULT LIST
# ============================================================

def print_compact_results(results, mode):
    for rank, result in enumerate(results, start=1):
        chunk = result["chunk"]

        if mode == "dense":
            score_text = (
                f"dense={result['dense_score']:.4f}"
            )

        else:
            score_text = (
                f"dense={result['dense_score']:.4f} | "
                f"rerank={result['reranker_score']:.4f}"
            )

        print(
            f"{rank:2}. "
            f"{chunk.get('ticker')} | "
            f"{chunk.get('content_type')} | "
            f"{score_text} | "
            f"global_rank={result.get('global_dense_rank')} | "
            f"table_rank={result.get('table_dense_rank')}"
        )

        print(
            chunk.get("section", "")
        )

        # Small preview
        text = chunk.get("text", "")
        print(text[:250].replace("\n", " "))
        print()


# ============================================================
# COMPARE DENSE VS RERANKED
# ============================================================

def compare_retrieval(
    query,
    global_top_k=30,
    table_top_k=10,
    final_top_k=10,
):
    # --------------------------------------------------------
    # 1. SAME candidate pool for both approaches
    # --------------------------------------------------------

    candidates, retrieval_time = retrieve_candidates(
        query=query,
        global_top_k=global_top_k,
        table_top_k=table_top_k,
    )

    # --------------------------------------------------------
    # 2. Pure dense ranking
    # --------------------------------------------------------

    dense_results = get_dense_top_k(
        candidates,
        top_k=final_top_k,
    )

    # --------------------------------------------------------
    # 3. Cross-encoder ranking
    # --------------------------------------------------------

    reranked_results, diagnostics = rerank_candidates(
        query=query,
        candidates=candidates,
        final_top_k=final_top_k,
        batch_size=32,
    )

    # --------------------------------------------------------
    # 4. Print comparison
    # --------------------------------------------------------

    print("\n" + "=" * 120)
    print("QUERY:")
    print(query)

    print("\n" + "=" * 120)
    print("PURE DENSE RETRIEVAL")
    print("=" * 120)

    print_compact_results(
        dense_results,
        mode="dense",
    )

    print("\n" + "=" * 120)
    print("DENSE + CROSS-ENCODER RERANKING")
    print("=" * 120)

    print_compact_results(
        reranked_results,
        mode="rerank",
    )

    print("=" * 120)
    print("TIMING")
    print("=" * 120)

    print(
        f"Dense candidate retrieval: "
        f"{retrieval_time:.3f}s"
    )

    print(
        f"Cross-encoder reranking: "
        f"{diagnostics['reranking_time']:.3f}s"
    )

    print(
        f"Candidates reranked: "
        f"{diagnostics['candidate_count']}"
    )

    print(
        f"Total reranker tokens: "
        f"{diagnostics['total_tokens']}"
    )

    return dense_results, reranked_results

In [120]:
company = "Mobileye"
test_case = test_queries[company][0]

query = test_case["query"]

dense_results, reranked_results = compare_retrieval(
    query=query,
    global_top_k=15,
    table_top_k=5,
    final_top_k=10,
)


QUERY:
What were Mobileye's total revenues in 2024?

PURE DENSE RETRIEVAL
 1. MBLY | narrative | dense=0.6548 | global_rank=1 | table_rank=None
Item 7 — Management’s Discussion and Analysis of Financial Condition and Results of Operations
Item 7 — Management’s Discussion and Analysis of Financial Condition and Results of Operations Company Overview  Mobileye is a leader in the development and deployment of ADAS and autonomous driving technologies and solutions. We pioneered ADAS techn

 2. MBLY | narrative | dense=0.6454 | global_rank=2 | table_rank=None
Item 8 — Financial Statements
Item 8 — Financial Statements 2024 Goodwill Impairment Test  During the third quarter of 2024, the Company performed an interim quantitative goodwill impairment analysis for the “Mobileye” reporting unit, due to a then-recent decline in the price of 

 3. MBLY | narrative | dense=0.6301 | global_rank=3 | table_rank=None
Item 7 — Management’s Discussion and Analysis of Financial Condition and Results of Op

In [ ]:
company = "Mobileye"
test_case = test_queries[company][0]

query = test_case["query"]

candidates, retrieval_time = retrieve_candidates(
    query=query,
    global_top_k=30,
    table_top_k=10,
)

print(f"Company:       {company}")
print(f"Expected type: {test_case['expected_type']}")
print(f"Query:         {query}")
print(f"Retrieval time: {retrieval_time:.3f}s\n")

print_pure_retrieval_results(
    candidates,
    top_k=10,
)

In [ ]:
reranked_results, diagnostics = rerank_candidates(
    query,
    candidates,
    final_top_k=10,
)

print(diagnostics)

print_results(reranked_results)